# This is an Incrementally Trained Modified-Huber Lossed Model on The US Accidents Data Set

**The Notebook is a part of the US Accidents Analysis Project**

**Link - https://www.kaggle.com/work/collections/18305074**

**This is the consensus of multiple training sessions**

The answers will be reported as the best discovered imbalanced classification report macro averages, 
but I looked at the full class wise report before picking the best model.

Do Note, the best discovered model has been retrained at the end; 
and the latest versions of the notebook will continue to report the same only. 
All the testing had been done privatelty.

Anyone interested in the model or any other steps if free to fork the notebook and try stuff out.
    
The **Modified-Huber Lossed** Model performerd reasonably okay, the best set of metrics I was able to reach is -
    
                       pre       rec       spe        f1       geo       iba       sup
    
              1      0.004     0.001     0.999     0.001     0.024     0.001     20220
              2      0.742     0.133     0.818     0.226     0.330     0.102   1846275
              3      0.199     0.097     0.921     0.130     0.299     0.082    390162
              4      0.027     0.771     0.227     0.051     0.418     0.184     61862
    
    avg / total      0.625     0.143     0.821     0.203     0.325     0.100   2318519

Achieved by - penalty='elasticnet', l1_ratio=0.8, alpha=9.95e-9

*Please Note, the Warnings have been specifically left on; so that the reader is aware of any poential changes or behaviour specifications.*

In [1]:
!git clone --filter=blob:none --no-checkout "https://github.com/Paras-GaurLRN/US-Accidents-EDA-Ensemble-Models.git"
%cd "US-Accidents-EDA-Ensemble-Models"
!git sparse-checkout init --cone
!git sparse-checkout set "notebooks/US Accidents - Pipelines/"
!git checkout main
%cd ..

Cloning into 'US-Accidents-EDA-Ensemble-Models'...
remote: Enumerating objects: 120, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 120 (delta 67), reused 75 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (120/120), 15.74 KiB | 3.15 MiB/s, done.
Resolving deltas: 100% (67/67), done.
/kaggle/working/US-Accidents-EDA-Ensemble-Models
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 12 (delta 1), reused 9 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (12/12), 17.92 MiB | 22.41 MiB/s, done.
Resolving deltas: 100% (1/1), done.
Updating files: 100% (12/12), done.
Already on 'main'
Your branch is up to date with 'origin/main'.
/kaggle/working


In [2]:
TRAIN_FILE = '/kaggle/input/notebooks/animeguylrn/us-accidents-models-preamble-work/US_Accidents_train.csv'
TEST_FILE = '/kaggle/input/notebooks/animeguylrn/us-accidents-models-preamble-work/US_Accidents_test.csv'
data_path = '/kaggle/working/US-Accidents-EDA-Ensemble-Models/notebooks/US Accidents - Pipelines'
transformers_path = '/kaggle/input/notebooks/animeguylrn/us-accidents-models-preamble-work/Transformers.pkl'

In [3]:
import sys

sys.path.append(data_path)

# To ensure that we can import the pipe

In [4]:
print("### Libraires ###\n")
with open(f'{data_path}/Libraries.txt') as LibrariesTXT:
    for line in LibrariesTXT.readlines():
        print(line)

### Libraires ###

scikit-learn

imbalanced-learn

feature-engine


In [5]:
print("### Imports ###\n")
with open(f'{data_path}/Imports.txt') as ImportsTXT:
    for line in ImportsTXT.readlines():
        print(line)

### Imports ###

from warnings import warn

from sklearn.base import (BaseEstimator, TransformerMixin, clone)

from sklearn.utils._param_validation import StrOptions

from imblearn.base import BaseSampler

from sklearn.utils.validation import check_is_fitted

from sklearn.compose import ColumnTransformer

from feature_engine.datetime import DatetimeFeatures

from feature_engine.outliers import ArbitraryOutlierCapper

from imblearn.pipeline import Pipeline as IMBPipe

import pandas as pd

import numpy as np


In [6]:
!pip install feature-engine

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 6.5 MB/s eta 0:00:00


In [7]:
!pip list | grep -E "numpy|pandas|feature-engine|imbalanced-learn|scikit-learn"

geopandas                                1.1.3
imbalanced-learn                         0.14.1
numpy                                    2.0.2
pandas                                   2.3.3
pandas-datareader                        0.10.0
pandas-gbq                               0.30.0
pandas-profiling                         3.6.6
pandas-stubs                             2.2.2.240909
pandasql                                 0.7.3
scikit-learn                             1.6.1
sklearn-pandas                           2.2.0


**Note: Target column = Severity**

# SGDClassifier

In [8]:
import numpy as np
import pandas as pd
from joblib import load, dump
from sklearn.preprocessing import (StandardScaler, OrdinalEncoder, OneHotEncoder)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from pipeline import (AnomalyCleaner, DateTimeFeatureEngineer, ColumnDropper, Illuminator, OutOfCoreNumericalImputer)

In [9]:
from sklearn.linear_model import SGDClassifier

**The Column Map**

*Pre-categories encoded for simplicity, OHE makes the final result a bit larger*
![Column Map](https://i.postimg.cc/zvyyYypX/Model-1.png)

In [10]:
from sklearn import set_config
set_config(transform_output='pandas')

# Loading The Transformers

In [11]:
Transformers = load(transformers_path)

In [12]:
for Transformer in Transformers.items():
    print(f'{Transformer[0]} : {Transformer[1]}',end='\n\n')

T1_AC : AnomalyCleaner()

T2_DTFE : DateTimeFeatureEngineer()

T3_IL : Illuminator()

T4_CD : ColumnDropper(columns=['Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction',
                       'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop',
                       'Traffic_Calming', 'Traffic_Signal', 'Start_Lat',
                       'Start_Lng', 'End_Lat', 'End_Lng', 'Distance(mi)',
                       'Street', 'City', 'Country', 'Timezone', 'Description',
                       'Zipcode', 'Airport_Code', 'Weather_Timestamp',
                       'Wind_Chill(F)'])

T5_SIM : ColumnTransformer(n_jobs=-1, remainder='passthrough',
                  transformers=[('num',
                                 OutOfCoreNumericalImputer(columns=['Temperature(F)',
                                                                    'Humidity(%)',
                                                                    'Pressure(in)',
                                                       

In [13]:
T1_AC = Transformers['T1_AC']
T2_DTFE = Transformers['T2_DTFE']
T3_IL = Transformers['T3_IL']
T4_CD = Transformers['T4_CD']
T5_SIM = Transformers['T5_SIM']
T6_ENC = Transformers['T6_ENC']
T7_SS = Transformers['T7_SS']

# Calculating Class Weights

In [14]:
classes = np.asarray([1,2,3,4])

y_counts = (
    pd.read_csv(TRAIN_FILE, usecols=["Severity"])["Severity"]
    .value_counts()
)

N = y_counts.sum()
K = len(y_counts)

class_weights = {
    cls: N / (K * count)
    for cls, count in y_counts.items()
}

# Model Training

In [15]:
SGDmodel = SGDClassifier(n_jobs=-1,verbose=0,shuffle=True,random_state=34,penalty='elasticnet',loss='modified_huber',
                         l1_ratio=0.80,
                         alpha=(9.95e-9))

first_iter = True
for chunk in pd.read_csv(TRAIN_FILE,
                         index_col='ID',
                         chunksize=20000):
    
    X, y = chunk.drop(columns=['Severity']), chunk['Severity']

    X, y = T1_AC.fit_resample(X, y)
    
    sample_weight = y.map(class_weights).to_numpy()
    
    X = T2_DTFE.transform(X)
    
    X = T3_IL.transform(X)
    
    X = T4_CD.transform(X)
    
    X = T5_SIM.transform(X)
    
    X = T6_ENC.transform(X)
    
    X = T7_SS.transform(X)

    if first_iter: SGDmodel.partial_fit(X, y, classes=classes, sample_weight=sample_weight); first_iter = False
    else: SGDmodel.partial_fit(X, y, sample_weight=sample_weight)

print("Model Trained!")

Model Trained!


# Model Performance

In [16]:
from imblearn.metrics import classification_report_imbalanced

test_set = pd.read_csv(TEST_FILE,index_col='ID')

X_pred, y_true = test_set.drop(columns=['Severity']), test_set['Severity']
del test_set

X = T1_AC.transform(X_pred, issue_warning=False)

X = T2_DTFE.transform(X)

X = T3_IL.transform(X)

X = T4_CD.transform(X)

X = T5_SIM.transform(X)

X = T6_ENC.transform(X)

X = T7_SS.transform(X)

y_pred = SGDmodel.predict(X)

print("Classification Report : ")
print(classification_report_imbalanced(y_true, y_pred, digits=3))

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Classification Report : 
                   pre       rec       spe        f1       geo       iba       sup

          1      0.004     0.001     0.999     0.001     0.024     0.001     20220
          2      0.742     0.133     0.818     0.226     0.330     0.102   1846275
          3      0.199     0.097     0.921     0.130     0.299     0.082    390162
          4      0.027     0.771     0.227     0.051     0.418     0.184     61862

avg / total      0.625     0.143     0.821     0.203     0.325     0.100   2318519



# Sinking The Model

In [17]:
dump(SGDmodel,'ModifiedHuberLossed.pkl')

['ModifiedHuberLossed.pkl']

In [18]:
# # Uncomment if you wish to see the coefficients of the model

# feature_names = SGDmodel.feature_names_in_

# coef_df = pd.DataFrame(
#     SGDmodel.coef_.T,
#     index=feature_names,
#     columns=[f"Class_{c}" for c in SGDmodel.classes_]
# )
# with pd.option_context(
#     "display.max_rows", None,
#     "display.max_columns", None,
#     "display.width", None,
#     "display.expand_frame_repr", False
# ):
#     print(coef_df.sort_values("Class_4", ascending=False).to_string(max_rows=None,max_cols=None))